# Phase 2 baseline training (Colab, free T4)

This notebook trains the Phase 2 baseline YOLOv8n detector on the merged
public dataset (car, bus, truck, motorcycle, autorickshaw, person,
crosswalk, signal_red, signal_green). It exists because this project's
dev machine has **no GPU**; training happens here instead, on a free
Colab T4, then the resulting `best.pt` is copied back into the repo's
`models/` directory.

**Before running this notebook**, produce the dataset zip locally and
upload it to Google Drive:

```bash
.venv/bin/python scripts/make_dataset_zip.py
# writes data/datasets/merged_dataset.zip (~220 MB)
```

Upload `merged_dataset.zip` to the root of your Google Drive (My Drive).
We upload a zip and unzip it on Colab instead of re-downloading from
Roboflow here, so this notebook never needs your Roboflow API key.

**Runtime:** make sure Runtime > Change runtime type > T4 GPU is selected
*before* running anything below. The next cell checks this for you and
will shout if it is wrong.

## Step 1 - confirm we actually have a GPU

Colab happily gives you a CPU-only runtime if you forget to change it, and
training will still start -- it will just take many hours instead of the
expected ~30-40 minutes for 60 epochs. `nvidia-smi` fails immediately (no
such command / no such device) if there is no GPU attached, so we check it
explicitly and refuse to continue rather than silently eating an entire
session on CPU.

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "NO GPU DETECTED. Go to Runtime > Change runtime type > select 'T4 GPU', "
        "then Runtime > Restart session, then re-run this notebook from the top. "
        "Training 60 epochs on CPU here will take many hours instead of ~30-40 minutes."
    )

print(result.stdout)
print("GPU detected, safe to proceed.")

## Step 2 - install ultralytics

Colab base images do not ship Ultralytics. This is the only extra
dependency we need; `torch` with CUDA support is already preinstalled on
the GPU runtime.

In [ ]:
!pip install -q ultralytics

## Step 3 - mount Drive and unzip the dataset

We mount Google Drive read-only from Colab's point of view (we only read
the zip), then unzip it into `/content/dataset`, which is local disk on
the Colab VM and much faster to train from than reading directly out of
Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Change this if you uploaded the zip somewhere other than Drive's root.
ZIP_PATH = "/content/drive/MyDrive/merged_dataset.zip"
DATASET_DIR = "/content/dataset"

import os
import zipfile

os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(DATASET_DIR)

# should show train/ and val/, each with images/ and labels/
print(os.listdir(DATASET_DIR))

## Step 4 - fix up data.yaml for Colab's paths

This is the step most likely to break if skipped. The `data/data.yaml`
committed in the repo has absolute paths from the dev machine
(`/home/parth/nft_project/data/datasets/merged/...`), which do not exist
on this VM. We do not edit the repo's data.yaml -- we write a fresh copy
here that points at `/content/dataset/train` and `/content/dataset/val`,
keeping the same 9 class names and IDs in the same order.

In [ ]:
CLASS_NAMES = [
    "car",
    "bus",
    "truck",
    "motorcycle",
    "autorickshaw",
    "person",
    "crosswalk",
    "signal_red",
    "signal_green",
]

COLAB_DATA_YAML = "/content/data.yaml"

with open(COLAB_DATA_YAML, "w") as f:
    f.write(f"train: {DATASET_DIR}/train/images\n")
    f.write(f"val: {DATASET_DIR}/val/images\n")
    f.write(f"nc: {len(CLASS_NAMES)}\n")
    f.write("names:\n")
    for name in CLASS_NAMES:
        f.write(f"  - {name}\n")

print(open(COLAB_DATA_YAML).read())

## Step 5 - train

Same call as `scripts/train.py` locally (`YOLO("yolov8n.pt").train(...)`),
just pointed at the Colab paths instead. Defaults match PLAN.md's Phase 2
baseline: 60 epochs, imgsz 640, batch 16. Known data gaps going in: `bus`
has almost no annotations and `signal_red`/`signal_green` have none at all
in the public data, so their mAP50 will be 0 or undefined -- that is the
expected "public data only" baseline, not a bug, and Phase 4 (our own
footage) is what fixes it.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=COLAB_DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=16,
    name="phase2_baseline",
)

## Step 6 - show the results

Ultralytics writes plots (PR curves, confusion matrix, sample batches) and
the `best.pt` / `last.pt` weights under `runs/detect/phase2_baseline/`.
We print the same per-class mAP50 table `scripts/train.py` prints locally,
so the two runs are directly comparable.

In [ ]:
class_names = results.names
per_class_map50 = {}
for i, class_id in enumerate(results.ap_class_index):
    name = class_names[int(class_id)]
    per_class_map50[name] = float(results.box.ap50[i])
for name in class_names.values():
    if name not in per_class_map50:
        per_class_map50[name] = None

print("per-class mAP50:")
for name, value in per_class_map50.items():
    value_str = f"{value:.4f}" if value is not None else "no val instances"
    print(f"  {name:<14} {value_str}")
print(f"\noverall mAP50:    {results.box.map50:.4f}")
print(f"overall mAP50-95: {results.box.map:.4f}")

In [ ]:
from IPython.display import Image, display

display(Image(filename="runs/detect/phase2_baseline/confusion_matrix.png"))
display(Image(filename="runs/detect/phase2_baseline/results.png"))

## Step 7 - zip up best.pt and the full runs/ output for download

We zip both the single weights file (what `src/detector.py` actually
needs) and the whole `runs/detect/phase2_baseline/` directory (PR curves,
confusion matrix, per-epoch metrics -- needed for the Phase 8 report).
Download the zip via the Colab file browser (left sidebar) and extract
`best.pt` into `models/` in the repo, and the rest into `report/`.

In [ ]:
import shutil

shutil.copy("runs/detect/phase2_baseline/weights/best.pt", "/content/best.pt")

shutil.make_archive("/content/phase2_results", "zip", "runs/detect/phase2_baseline")

print("download these two files from the Colab file browser:")
print("  /content/best.pt              -> copy to models/best.pt in the repo")
print("  /content/phase2_results.zip   -> unzip into report/phase2_baseline_run/")

In [ ]:
from google.colab import files

files.download("/content/best.pt")
files.download("/content/phase2_results.zip")